In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import pickle

In [62]:
df = pd.read_csv('social_media_screentime_mental_health_2026.csv')
# df.head(10)
df.isnull().sum()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 25 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   participant_id                       7000 non-null   object 
 1   age                                  7000 non-null   int64  
 2   gender                               6930 non-null   object 
 3   occupation                           7000 non-null   object 
 4   region                               7000 non-null   object 
 5   most_used_platform                   7000 non-null   object 
 6   platforms_used_count                 7000 non-null   int64  
 7   daily_screen_hours                   7000 non-null   float64
 8   daily_notifications                  7000 non-null   int64  
 9   night_time_use                       7000 non-null   object 
 10  minutes_to_first_check_after_waking  7000 non-null   int64  
 11  primary_purpose               

In [63]:
#FILLING NULL VALUES
df['gender'] = df['gender'].fillna('Unknown')
# df.head(99)

df['avg_sleep_hours'] = df['avg_sleep_hours'].fillna(df['avg_sleep_hours'].median())
df.isnull().sum()


participant_id                         0
age                                    0
gender                                 0
occupation                             0
region                                 0
most_used_platform                     0
platforms_used_count                   0
daily_screen_hours                     0
daily_notifications                    0
night_time_use                         0
minutes_to_first_check_after_waking    0
primary_purpose                        0
avg_sleep_hours                        0
anxiety_score_0to27                    0
low_mood_score_0to27                   0
life_satisfaction_1to10                0
loneliness_1to10                       0
self_esteem_1to10                      0
fomo_1to10                             0
social_comparison_1to10                0
physical_activity_days_per_week        0
uses_screen_time_limits                0
attempted_digital_detox                0
seeks_mental_health_support            0
wellbeing_band  

In [64]:
X = df.drop(columns=['participant_id','wellbeing_band'])
y = df['wellbeing_band']


In [65]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train

,age,gender,occupation,region,most_used_platform,platforms_used_count,daily_screen_hours,daily_notifications,night_time_use,minutes_to_first_check_after_waking,...,low_mood_score_0to27,life_satisfaction_1to10,loneliness_1to10,self_esteem_1to10,fomo_1to10,social_comparison_1to10,physical_activity_days_per_week,uses_screen_time_limits,attempted_digital_detox,seeks_mental_health_support
1032,15,Female,Full-time employed,Europe,Instagram,4,1.7,63,Often,24,...,9,7,2,8,6,3,2,No,"Yes, succeeded",Considering it
6339,13,Male,Unemployed,Asia,Facebook,6,2.9,41,Sometimes,6,...,0,9,4,7,6,7,2,No,No,Considering it
3886,20,Male,Student,North America,YouTube,1,6.4,303,Often,13,...,0,7,6,4,3,8,3,No,No,No
2653,37,Female,Student,Asia,Instagram,4,6.0,89,Every night,2,...,0,5,4,2,9,5,4,No,"Yes, failed",Considering it
6914,29,Male,Student,Europe,Reddit,2,6.8,104,Sometimes,13,...,0,7,8,7,8,7,2,Yes,"Yes, succeeded",Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,15,Male,Full-time employed,North America,Reddit,4,4.0,25,Sometimes,14,...,1,6,7,3,3,3,4,No,No,No
5191,34,Female,Student,Africa,YouTube,3,1.1,17,Never,10,...,7,9,3,7,2,6,1,No,"Yes, failed",Yes
5226,42,Male,Student,North America,TikTok,4,6.7,97,Never,7,...,3,4,6,9,8,8,3,Yes,"Yes, succeeded",No
5390,26,Male,Full-time employed,North America,YouTube,2,4.5,22,Sometimes,27,...,7,6,3,6,4,7,4,Yes,No,Yes


In [70]:
# Identify feature types
numeric_features = X.select_dtypes(include=['int64','float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Preprocessing
numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Build pipeline with preprocessing + model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [71]:

model.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['age', 'platforms_used_count', 'daily_screen_hours',
       'daily_notifications', 'minutes_to_first_check_after_waking',
       'avg_sleep_hours', 'anxiety_score_0to27', 'low_mood_score_0to27',
       'life_satisfaction_1to10', 'loneliness_1to10', 'self_esteem_1to10',
       '...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'occupation', 'region', 'most_used_platform',
       'night_time_use', 'primary_purpose', 'uses_screen_time_limits',
       'attempted_digital_detox', 'seeks_mental_health_support'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [72]:
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Load model back
with open('rf_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [73]:
# Predict on test set
y_pred = loaded_model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9964285714285714


In [74]:

# Example prediction on a single new row (must match feature order!)
sample = pd.DataFrame([{
    'age': 25,
    'gender': 'Female',
    'occupation': 'Student',
    'region': 'Urban',
    'most_used_platform': 'Instagram',
    'platforms_used_count': 3,
    'daily_screen_hours': 5.5,
    'daily_notifications': 120,
    'night_time_use': 'Yes',
    'minutes_to_first_check_after_waking': 10,
    'primary_purpose': 'Entertainment',
    'avg_sleep_hours': 7.0,
    'anxiety_score_0to27': 12,
    'low_mood_score_0to27': 8,
    'life_satisfaction_1to10': 7,
    'loneliness_1to10': 6,
    'self_esteem_1to10': 8,
    'fomo_1to10': 5,
    'social_comparison_1to10': 4,
    'physical_activity_days_per_week': 3,
    'uses_screen_time_limits': 'Yes',
    'attempted_digital_detox': 'No',
    'seeks_mental_health_support': 'Yes'
}])

print("Predicted wellbeing band:", loaded_model.predict(sample))

Predicted wellbeing band: ['Moderate']
